# Fairness Audit & Sample Reweighting Mitigation - Project 2
> **Production Alignment Note:** Evaluates both unmitigated and sample-reweighted `XGBClassifier` models at the validation-calibrated unified decision threshold (`0.1358` from `artifacts/decision_thresholds.json`) and computes 95% Katz confidence intervals (`src/evaluation/evaluate.py`).

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation.evaluate import _cohort_confusion_rates, _katz_95_ci
from src.training.train import LoanModelTrainer

train_df = pd.read_parquet(PROJECT_ROOT / "data/processed/train.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "data/processed/test.parquet")
thresh_file = PROJECT_ROOT / "artifacts/decision_thresholds.json"
DECISION_THRESHOLD = (
    float(json.loads(thresh_file.read_text())["decision_threshold"])
    if thresh_file.is_file()
    else 0.1358
)
print(f"Evaluating at validation-calibrated threshold: {DECISION_THRESHOLD}")

In [ ]:
def encode_loan_frame(df_in: pd.DataFrame):
    is_young = (df_in["Age"] < 30).astype(int)
    drop_cols = [
        c
        for c in ("LoanID", "Default", "MaritalStatus", "is_young")
        if c in df_in.columns
    ]
    X = pd.get_dummies(df_in.drop(columns=drop_cols), drop_first=True)
    X = X.astype({col: "int8" for col in X.select_dtypes(include=["bool"]).columns})
    return X, df_in["Default"], is_young


X_train, y_train, _ = encode_loan_frame(train_df)
X_test, y_test, is_young_test = encode_loan_frame(test_df)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)


def audit_model_fairness(
    model, X_eval, y_eval, is_young_eval, cutoff: float = DECISION_THRESHOLD
):
    probs = model.predict_proba(X_eval)[:, 1]
    preds = pd.Series((probs >= cutoff).astype(int), index=X_eval.index)
    prot = _cohort_confusion_rates(
        y_eval[is_young_eval == 1], preds[is_young_eval == 1]
    )
    base = _cohort_confusion_rates(
        y_eval[is_young_eval == 0], preds[is_young_eval == 0]
    )
    di = (
        round(prot["approval_rate"] / base["approval_rate"], 4)
        if base["approval_rate"] > 0
        else 0.0
    )
    ci = _katz_95_ci(
        prot["approved_count"], prot["count"], base["approved_count"], base["count"]
    )
    return {
        "cutoff": cutoff,
        "di_ratio": di,
        "di_95_ci": ci,
        "under_30_approval": prot["approval_rate"],
        "over_30_approval": base["approval_rate"],
    }

In [ ]:
baseline_clf = XGBClassifier(random_state=42, eval_metric="logloss", n_estimators=100)
baseline_clf.fit(X_train, y_train)
print(
    "Unmitigated Baseline @ 0.1358:",
    audit_model_fairness(baseline_clf, X_test, y_test, is_young_test),
)

trainer = LoanModelTrainer(data_dir=PROJECT_ROOT / "data/processed")
weights = trainer.calculate_mitigation_weights(train_df)
fair_clf = XGBClassifier(random_state=42, eval_metric="logloss", n_estimators=100)
fair_clf.fit(X_train, y_train, sample_weight=weights)
print(
    "Sample-Reweighted Model @ 0.1358:",
    audit_model_fairness(fair_clf, X_test, y_test, is_young_test),
)